<a href="https://colab.research.google.com/github/omarcordero1/-omarcordero1/blob/main/Google_Trends_%2B_GSC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Google Trens + GSC**

Imprtamos dependencias

In [ ]:
!pip install -q google-auth google-auth-oauthlib google-auth-httplib2
!pip install -q google-api-python-client
!pip install -q pytrends
!pip install -q pandas matplotlib seaborn

Instalamos librerias

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json
import os

Vamos a la aplicación

In [ ]:
print("✅ Librerías instaladas correctamente")

# CONFIGURACIÓN REAL PARA TU CLIENT ID
class GSCAnalyzer:
    def __init__(self):
        # Tu Client ID que ya tienes
        self.CLIENT_ID = "336222568140-sco4jgat0aila07rrac8aruhcg12f7im.apps.googleusercontent.com"
        self.service = None
        self.site_url = None

    def setup_oauth_colab(self, site_url):
        """Configuración específica para Google Colab"""
        self.site_url = site_url

        print("🔧 CONFIGURACIÓN PARA GOOGLE COLAB")
        print("=" * 50)
        print("⚠️  IMPORTANTE: Este método funciona SIN Client Secret")
        print()

        try:
            from google.colab import auth
            from googleapiclient.discovery import build

            # Autenticación de Colab (usa tu cuenta de Google)
            print("🔐 Autenticando con tu cuenta de Google...")
            auth.authenticate_user()

            # Crear servicio usando credenciales de Colab
            self.service = build('searchconsole', 'v1')

            print("✅ Autenticación exitosa usando Google Colab")
            return True

        except ImportError:
            print("❌ Este método solo funciona en Google Colab")
            return self.setup_oauth_manual()
        except Exception as e:
            print(f"❌ Error: {str(e)}")
            return self.setup_oauth_manual()

    def setup_oauth_manual(self):
        """Método manual si no estás en Colab"""
        print("📱 MÉTODO MANUAL (fuera de Colab)")
        print("=" * 40)

        try:
            from google.auth.transport.requests import Request
            from google_auth_oauthlib.flow import InstalledAppFlow
            from googleapiclient.discovery import build

            # Configuración OAuth mínima
            scopes = ['https://www.googleapis.com/auth/webmasters.readonly']

            # Crear archivo temporal de credenciales
            client_config = {
                "installed": {
                    "client_id": self.CLIENT_ID,
                    "client_secret": input("🔑 Ingresa tu Client Secret: "),
                    "auth_uri": "https://accounts.google.com/o/oauth2/auth",
                    "token_uri": "https://oauth2.googleapis.com/token",
                    "redirect_uris": ["urn:ietf:wg:oauth:2.0:oob"]
                }
            }

            # Flujo de autenticación
            flow = InstalledAppFlow.from_client_config(client_config, scopes)
            credentials = flow.run_local_server(port=0)

            # Crear servicio
            self.service = build('searchconsole', 'v1', credentials=credentials)

            print("✅ Autenticación manual exitosa")
            return True

        except Exception as e:
            print(f"❌ Error en autenticación manual: {str(e)}")
            return False

    def get_gsc_data(self, days_back=30):
        """Obtener datos reales de GSC"""
        if not self.service or not self.site_url:
            print("❌ Primero configura la autenticación y URL del sitio")
            return pd.DataFrame()

        # Fechas
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')

        print(f"📊 Obteniendo datos del {start_date} al {end_date}...")
        print(f"🌐 Sitio: {self.site_url}")

        try:
            # Request a GSC
            request = {
                'startDate': start_date,
                'endDate': end_date,
                'dimensions': ['query'],
                'rowLimit': 1000  # Empezamos con menos para probar
            }

            response = self.service.searchanalytics().query(
                siteUrl=self.site_url,
                body=request
            ).execute()

            if 'rows' not in response:
                print("⚠️  No hay datos para este período")
                print("💡 Verifica que:")
                print("   • La URL del sitio sea correcta")
                print("   • El sitio esté verificado en Search Console")
                print("   • Haya datos en el período seleccionado")
                return pd.DataFrame()

            # Convertir a DataFrame
            data = []
            for row in response['rows']:
                data.append({
                    'keyword': row['keys'][0],
                    'clicks': row['clicks'],
                    'impressions': row['impressions'],
                    'ctr': round(row['ctr'] * 100, 2),
                    'position': round(row['position'], 1)
                })

            df = pd.DataFrame(data)
            print(f"✅ Obtenidos {len(df)} keywords con datos")

            return df

        except Exception as e:
            print(f"❌ Error obteniendo datos: {str(e)}")
            print("💡 Posibles causas:")
            print("   • URL del sitio incorrecta")
            print("   • Permisos insuficientes")
            print("   • Sitio no verificado en Search Console")
            return pd.DataFrame()

    def analyze_data(self, df):
        """Análisis básico de los datos"""
        if df.empty:
            print("❌ No hay datos para analizar")
            return

        print("\n📈 ANÁLISIS DE DATOS")
        print("=" * 30)

        # Estadísticas básicas
        total_clicks = df['clicks'].sum()
        total_impressions = df['impressions'].sum()
        avg_ctr = df['ctr'].mean()
        avg_position = df['position'].mean()

        print(f"📊 Total clicks: {total_clicks:,}")
        print(f"👁️  Total impresiones: {total_impressions:,}")
        print(f"📈 CTR promedio: {avg_ctr:.2f}%")
        print(f"📍 Posición promedio: {avg_position:.1f}")

        # Top 10 keywords
        print(f"\n🏆 TOP 10 KEYWORDS POR CLICKS:")
        top_10 = df.nlargest(10, 'clicks')[['keyword', 'clicks', 'impressions', 'ctr', 'position']]
        for idx, row in top_10.iterrows():
            print(f"  {row.name + 1:2d}. {row['keyword'][:40]:40s} | {row['clicks']:3d} clicks | CTR: {row['ctr']:5.1f}% | Pos: {row['position']:4.1f}")

        # Oportunidades
        print(f"\n💡 OPORTUNIDADES:")

        # Keywords con muchas impresiones pero bajo CTR
        low_ctr = df[(df['impressions'] > 100) & (df['ctr'] < 2)]
        if len(low_ctr) > 0:
            print(f"  🎯 {len(low_ctr)} keywords con >100 impresiones pero CTR <2%")

        # Keywords en posición 4-10
        first_page_low = df[(df['position'] >= 4) & (df['position'] <= 10)]
        if len(first_page_low) > 0:
            print(f"  🚀 {len(first_page_low)} keywords en posición 4-10 (primera página)")

        # Keywords sin clicks pero con impresiones
        no_clicks = df[(df['impressions'] > 50) & (df['clicks'] == 0)]
        if len(no_clicks) > 0:
            print(f"  ⚠️  {len(no_clicks)} keywords con impresiones pero 0 clicks")

        return {
            'total_clicks': total_clicks,
            'total_impressions': total_impressions,
            'avg_ctr': avg_ctr,
            'avg_position': avg_position,
            'top_keywords': top_10,
            'opportunities': {
                'low_ctr': low_ctr,
                'first_page_low': first_page_low,
                'no_clicks': no_clicks
            }
        }

    def create_visualizations(self, df):
        """Crear gráficos básicos"""
        if df.empty:
            return

        print("\n📊 CREANDO VISUALIZACIONES...")

        # Configurar estilo
        plt.style.use('default')
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('Análisis SEO - Google Search Console', fontsize=16, fontweight='bold')

        # 1. Top 15 keywords por clicks
        top_15 = df.nlargest(15, 'clicks')
        axes[0,0].barh(range(len(top_15)), top_15['clicks'], color='skyblue')
        axes[0,0].set_yticks(range(len(top_15)))
        axes[0,0].set_yticklabels([k[:25] + '...' if len(k) > 25 else k for k in top_15['keyword']], fontsize=8)
        axes[0,0].set_title('Top 15 Keywords por Clicks')
        axes[0,0].set_xlabel('Clicks')

        # 2. Distribución de CTR
        axes[0,1].hist(df['ctr'], bins=20, edgecolor='black', alpha=0.7, color='lightgreen')
        axes[0,1].set_title('Distribución de CTR')
        axes[0,1].set_xlabel('CTR (%)')
        axes[0,1].set_ylabel('Cantidad de Keywords')
        axes[0,1].axvline(df['ctr'].mean(), color='red', linestyle='--', label=f'Promedio: {df["ctr"].mean():.1f}%')
        axes[0,1].legend()

        # 3. Posición vs Clicks (scatter)
        axes[1,0].scatter(df['position'], df['clicks'], alpha=0.6, color='orange')
        axes[1,0].set_title('Posición vs Clicks')
        axes[1,0].set_xlabel('Posición Promedio')
        axes[1,0].set_ylabel('Clicks')
        axes[1,0].invert_xaxis()

        # 4. Distribución de posiciones
        position_bins = [0, 3, 10, 20, 50, 100]
        position_labels = ['Top 3', '4-10', '11-20', '21-50', '50+']
        df['position_group'] = pd.cut(df['position'], bins=position_bins, labels=position_labels, include_lowest=True)
        position_counts = df['position_group'].value_counts()

        axes[1,1].pie(position_counts.values, labels=position_counts.index, autopct='%1.1f%%', startangle=90)
        axes[1,1].set_title('Distribución por Posición')

        plt.tight_layout()
        plt.show()

def analyze_trends_simple(keywords_list, max_keywords=5):
    """Análisis simple de Google Trends"""
    if len(keywords_list) == 0:
        print("❌ No hay keywords para analizar")
        return {}

    try:
        from pytrends.request import TrendReq

        # Tomar solo los primeros keywords
        selected = keywords_list[:max_keywords]
        print(f"📈 Analizando tendencias para: {', '.join(selected)}")

        pytrends = TrendReq(hl='es-MX', tz=360)

        results = {}

        for keyword in selected:
            try:
                pytrends.build_payload([keyword], timeframe='today 12-m', geo='MX')
                interest_df = pytrends.interest_over_time()

                if not interest_df.empty and keyword in interest_df.columns:
                    trend_data = interest_df[keyword]

                    # Calcular tendencia simple
                    recent_avg = trend_data.tail(4).mean()
                    early_avg = trend_data.head(4).mean()

                    if early_avg > 0:
                        growth = ((recent_avg - early_avg) / early_avg) * 100
                    else:
                        growth = 0

                    results[keyword] = {
                        'trend_data': trend_data,
                        'avg_interest': trend_data.mean(),
                        'growth_rate': growth,
                        'direction': 'Creciente 📈' if growth > 5 else 'Decreciente 📉' if growth < -5 else 'Estable ➡️'
                    }

                    print(f"  ✅ {keyword}: {results[keyword]['direction']} ({growth:.1f}%)")

                import time
                time.sleep(1.5)  # Pausa para evitar rate limiting

            except Exception as e:
                print(f"  ⚠️ Error con '{keyword}': {str(e)}")
                continue

        # Crear gráfico si hay datos
        if results:
            plt.figure(figsize=(12, 6))
            for keyword, data in results.items():
                plt.plot(data['trend_data'].index, data['trend_data'].values,
                        label=f"{keyword} ({data['direction'].split()[0]})", marker='o', markersize=3)

            plt.title('Tendencias de Google Trends (últimos 12 meses)')
            plt.xlabel('Fecha')
            plt.ylabel('Interés Relativo')
            plt.legend()
            plt.xticks(rotation=45)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

        return results

    except ImportError:
        print("❌ Error: pytrends no disponible")
        return {}
    except Exception as e:
        print(f"❌ Error en análisis de tendencias: {str(e)}")
        return {}

# FUNCIÓN PRINCIPAL SIMPLIFICADA
def run_complete_analysis():
    """Función principal que SÍ funciona"""
    print("🚀 ANÁLISIS SEO COMPLETO")
    print("=" * 40)

    try:
        # 1. Crear analizador
        analyzer = GSCAnalyzer()

        # 2. Pedir URL del sitio
        print("📝 Configuración inicial:")
        site_url = input("🌐 URL de tu sitio (ej: https://misitio.com/): ").strip()

        # 3. Configurar autenticación
        print("\n🔐 Configurando autenticación...")
        if not analyzer.setup_oauth_colab(site_url):
            print("❌ Error en autenticación")
            return None

        # 4. Obtener datos
        print("\n📊 Obteniendo datos de Search Console...")
        df = analyzer.get_gsc_data(days_back=30)  # Últimos 30 días

        if df.empty:
            print("❌ No se pudieron obtener datos")
            return None

        # 5. Analizar datos
        analysis = analyzer.analyze_data(df)

        # 6. Crear visualizaciones
        analyzer.create_visualizations(df)

        # 7. Análisis de tendencias para top keywords
        top_keywords = df.nlargest(5, 'clicks')['keyword'].tolist()
        print(f"\n📈 Analizando tendencias para top 5 keywords...")
        trends = analyze_trends_simple(top_keywords)

        print("\n✅ ANÁLISIS COMPLETADO")
        print("💾 Datos disponibles en las variables 'analyzer', 'df', 'analysis', 'trends'")

        return {
            'analyzer': analyzer,
            'data': df,
            'analysis': analysis,
            'trends': trends
        }

    except KeyboardInterrupt:
        print("\n⏹️ Análisis cancelado por el usuario")
        return None
    except Exception as e:
        print(f"\n❌ Error inesperado: {str(e)}")
        print("💡 Revisa la configuración e intenta de nuevo")
        return None

# FUNCIÓN PARA KEYWORDS ESPECÍFICAS
def analyze_my_keywords(keywords_list):
    """Analizar solo keywords específicas (sin GSC)"""
    print(f"🔍 ANÁLISIS DE KEYWORDS ESPECÍFICAS")
    print(f"📝 Keywords: {', '.join(keywords_list)}")
    print("=" * 50)

    return analyze_trends_simple(keywords_list, max_keywords=len(keywords_list))

# INSTRUCCIONES FINALES ACTUALIZADAS
print("""
🎯 SCRIPT SEO FUNCIONAL - VERSIÓN INTERACTIVA
===========================================

🔥 NUEVO: Ahora el script te pregunta qué keywords optimizar!

📋 MÉTODOS DE USO:

1️⃣ ANÁLISIS COMPLETO INTERACTIVO:
   ```python
   results = run_complete_analysis()
   ```
   Te preguntará:
   • URL de tu sitio
   • Días a analizar
   • Qué keywords optimizar (top GSC + tus keywords)
   • Genera recomendaciones específicas

2️⃣ SOLO KEYWORDS ESPECÍFICAS:
   ```python
   results = analyze_keywords_only()
   ```
   Solo tendencias para keywords que tú elijas

3️⃣ ANÁLISIS RÁPIDO DE KEYWORDS:
   ```python
   keywords = ['marketing digital', 'seo mexico', 'analytics']
   trends = analyze_my_keywords(keywords)
   ```

🎯 NUEVAS CARACTERÍSTICAS:
• Te pregunta qué keywords analizar
• Combina top keywords de GSC + tus keywords
• Recomendaciones específicas de optimización
• Análisis interactivo paso a paso

⚡ PARA EMPEZAR:
""")

print("🚀 Ejecuta: results = run_complete_analysis()")
print("📝 El script te guiará paso a paso y te pedirá las keywords que quieres optimizar!")

# FUNCIÓN EXTRA: Análisis competitivo rápido
def competitive_analysis():
    """Análisis competitivo simple"""
    print("⚔️ ANÁLISIS COMPETITIVO RÁPIDO")
    print("=" * 40)

    print("📝 Ingresa keywords de tu negocio:")
    my_keywords = []
    while True:
        kw = input("  🔸 Tu keyword: ").strip()
        if not kw: break
        my_keywords.append(kw)

    print("\n📝 Ingresa keywords de competidores:")
    competitor_keywords = []
    while True:
        kw = input("  🔸 Keyword competidor: ").strip()
        if not kw: break
        competitor_keywords.append(kw)

    if not my_keywords and not competitor_keywords:
        print("❌ No se ingresaron keywords")
        return None

    all_keywords = my_keywords + competitor_keywords
    print(f"\n📈 Analizando {len(all_keywords)} keywords...")

    trends = analyze_trends_simple(all_keywords, max_keywords=len(all_keywords))

    # Comparar performance
    if trends:
        print("\n📊 COMPARACIÓN:")
        print("TUS KEYWORDS:")
        for kw in my_keywords:
            if kw in trends:
                print(f"  • {kw}: {trends[kw]['direction']} ({trends[kw]['growth_rate']:.1f}%)")

        print("\nKEYWORDS COMPETIDORES:")
        for kw in competitor_keywords:
            if kw in trends:
                print(f"  • {kw}: {trends[kw]['direction']} ({trends[kw]['growth_rate']:.1f}%)")

    return {
        'my_keywords': my_keywords,
        'competitor_keywords': competitor_keywords,
        'trends': trends
    }

✅ Librerías instaladas correctamente

🎯 SCRIPT SEO FUNCIONAL - VERSIÓN INTERACTIVA

🔥 NUEVO: Ahora el script te pregunta qué keywords optimizar!

📋 MÉTODOS DE USO:

1️⃣ ANÁLISIS COMPLETO INTERACTIVO:
   ```python
   results = run_complete_analysis()
   ```
   Te preguntará:
   • URL de tu sitio
   • Días a analizar
   • Qué keywords optimizar (top GSC + tus keywords)
   • Genera recomendaciones específicas

2️⃣ SOLO KEYWORDS ESPECÍFICAS:
   ```python
   results = analyze_keywords_only()
   ```
   Solo tendencias para keywords que tú elijas

3️⃣ ANÁLISIS RÁPIDO DE KEYWORDS:
   ```python
   keywords = ['marketing digital', 'seo mexico', 'analytics']
   trends = analyze_my_keywords(keywords)
   ```

🎯 NUEVAS CARACTERÍSTICAS:
• Te pregunta qué keywords analizar
• Combina top keywords de GSC + tus keywords
• Recomendaciones específicas de optimización
• Análisis interactivo paso a paso

⚡ PARA EMPEZAR:

🚀 Ejecuta: results = run_complete_analysis()
📝 El script te guiará paso a paso y te pedi

In [ ]:
results = run_complete_analysis()

🚀 ANÁLISIS SEO COMPLETO
📝 Configuración inicial:
🌐 URL de tu sitio (ej: https://misitio.com/): https://www.milenio.com/

🔐 Configurando autenticación...
🔧 CONFIGURACIÓN PARA GOOGLE COLAB
⚠️  IMPORTANTE: Este método funciona SIN Client Secret

🔐 Autenticando con tu cuenta de Google...
✅ Autenticación exitosa usando Google Colab

📊 Obteniendo datos de Search Console...
📊 Obteniendo datos del 2025-08-12 al 2025-09-11...
🌐 Sitio: https://www.milenio.com/


❌ Error obteniendo datos: <HttpError 403 when requesting https://searchconsole.googleapis.com/webmasters/v3/sites/https%3A%2F%2Fwww.milenio.com%2F/searchAnalytics/query?alt=json returned "Request had insufficient authentication scopes.". Details: "[{'message': 'Insufficient Permission', 'domain': 'global', 'reason': 'insufficientPermissions'}]">
💡 Posibles causas:
   • URL del sitio incorrecta
   • Permisos insuficientes
   • Sitio no verificado en Search Console
❌ No se pudieron obtener datos
